# Feature Engineering for Depression Prediction


In [14]:
import numpy as np
import pandas as pd


In [18]:
df = pd.read_csv("data_final.csv")
target_col = "Depression"

print("Shape:", df.shape)
display(df.head(3))



Shape: (140608, 20)


,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,0.0,5.0,-1.00,0.0,2.0,5,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,0.0,4.0,-1.00,0.0,3.0,2,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,Student,5.0,0.0,8.97,2.0,0.0,2,Healthy,B.Pharm,Yes,3.0,1.0,No,1


- Theo phân tích từ câu 1 của mục Data Analyst thì tiến hành tạo ra feature bad_sleep, bad_work, bad_diet và cuối cùng là bad_habits_count vì khi phân tích từ dữ liệu có thể thấy có xu hướng giữa khả năng mắc bệnh trầm cảm nếu có nhiều thói quen xấu

- Theo phân tích từ câu 2 các nhóm độ tuổi trẻ thường có khả năng cao mắc bệnh trầm cảm và có xu hướng giảm khi độ tuổi càng tăng nên tiến hành chia nhóm tuổi ra thành 5 mốc độ tuổi: 18-25, 26-30, 31-35, 36-45, 46+ 




In [ ]:
def create_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    # Binning Work/Study Hours
    bins = [0, 5, 8, 12]
    labels = ['Low', 'Medium', 'High']
    d['Work/Study Hours'] = pd.cut(
        d['Work/Study Hours'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # bad habit flags
    sleep_num = pd.to_numeric(d["Sleep Duration"], errors="coerce")
    d["bad_sleep"] = (sleep_num <= 2).astype("Int64")
    d["bad_work"] = (d["Work/Study Hours"].astype(str).eq("High")).astype("Int64")
    d["bad_diet"] = (d["Dietary Habits"].astype(str).eq("Unhealthy")).astype("Int64")
    d["bad_habits_count"] = d[["bad_sleep", "bad_work", "bad_diet"]].fillna(0).sum(axis=1)

    # age grouping
    d["Age_Group"] = pd.cut(
        d["Age"],
        bins=[18, 25, 30, 35, 45, 60],
        labels=["18-25", "26-30", "31-35", "36-45", "46+"],
        include_lowest=True,
    )

    # interaction effects
    d["Diet_Work_Interaction"] = (
        d["Dietary Habits"].astype(str) + "__" + d["Work/Study Hours"].astype(str)
    )
    d["Sleep_Work_Interaction"] = (
        d["Sleep Duration"].astype(str) + "__" + d["Work/Study Hours"].astype(str)
    )

    return d


# apply
df_eng = create_engineered_features(df)

print("Engineered shape:", df_eng.shape)
display(df_eng.head(3))

Engineered shape: (140608, 27)


,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,...,Financial Stress,Family History of Mental Illness,Depression,bad_sleep,bad_work,bad_diet,bad_habits_count,Age_Group,Diet_Work_Interaction,Sleep_Work_Interaction
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,0.0,5.0,-1.00,...,2.0,No,0,0,0,0,0,46+,Healthy__Low,5__Low
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,0.0,4.0,-1.00,...,3.0,No,1,1,0,1,2,26-30,Unhealthy__Medium,2__Medium
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,Student,5.0,0.0,8.97,...,1.0,No,1,1,0,0,1,31-35,Healthy__Low,2__Low


- Theo phân tích từ câu 2 => giới tính không ảnh hưởng đặc biệt đến khả năng bị trầm cảm nên tiến hành xóa ddi feature này.

- Theo phân tích từ câu 1 tiến hành xóa đi cột sleep_ vì khi phân tích không thấy sự ảnh hưởng của feature này lên khả năng bị bệnh trầm cảm.


In [28]:
target_col = "Depression"
drop_always = ["id", "Name", "Gender", "Sleep Duration", "Degree"]

def build_dataset(data: pd.DataFrame, extra_drop=None):
    extra_drop = extra_drop or []
    
    use = data.drop(columns=drop_always + extra_drop, errors="ignore")
    y = data[target_col].astype(int)
    
    return use, y

X_base, y = build_dataset(df)
X_eng, y = build_dataset(df_eng)

print("Baseline feature count:", X_base.shape[1])
print("Engineered feature count:", X_eng.shape[1])

Baseline feature count: 15
Engineered feature count: 22


In [30]:
X_eng.head(3)

,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Dietary Habits,...,Financial Stress,Family History of Mental Illness,Depression,bad_sleep,bad_work,bad_diet,bad_habits_count,Age_Group,Diet_Work_Interaction,Sleep_Work_Interaction
0,49.0,Ludhiana,Working Professional,Chef,0.0,5.0,-1.00,0.0,2.0,Healthy,...,2.0,No,0,0,0,0,0,46+,Healthy__Low,5__Low
1,26.0,Varanasi,Working Professional,Teacher,0.0,4.0,-1.00,0.0,3.0,Unhealthy,...,3.0,No,1,1,0,1,2,26-30,Unhealthy__Medium,2__Medium
2,33.0,Visakhapatnam,Student,Student,5.0,0.0,8.97,2.0,0.0,Healthy,...,1.0,No,1,1,0,0,1,31-35,Healthy__Low,2__Low
